## `CMD` vs `ENTRYPOINT`

The two instructions everyone gets wrong at first. Both set what the container runs by default; the difference is how they **combine** and how run-time arguments override them.

- **`CMD`** sets the **default arguments** — easily overridden by anything you pass to `docker run`.
- **`ENTRYPOINT`** sets the **fixed executable** — run-time arguments are *appended* to it.

The mental model:

```
final command = ENTRYPOINT + (run-time args OR CMD)
```

Arguments in `docker run img foo bar` *replace* `CMD` but are *appended* to `ENTRYPOINT`.

| Dockerfile | `docker run img` | `docker run img foo bar` |
|---|---|---|
| `CMD ["python","app.py"]` | `python app.py` | `foo bar` |
| `ENTRYPOINT ["python","app.py"]` | `python app.py` | `python app.py foo bar` |
| `ENTRYPOINT ["python"]` + `CMD ["app.py"]` | `python app.py` | `python foo bar` |
| `CMD echo hi` (shell form) | `/bin/sh -c "echo hi"` | `foo bar` (shell-form CMD dropped wholesale) |

**Practical rules:**

- **A tool that takes flags** (think `curl`, `git`, `aws`) — `ENTRYPOINT` for the program, `CMD` for default args. Then `docker run img --version` "just works."
- **A long-lived service configured by env vars** (a web app, a daemon) — just `CMD ["python","app.py"]`, no `ENTRYPOINT`, so a user can drop into a shell with `docker run -it img bash`.

**Always prefer exec form** (the JSON-array `["python","app.py"]`) over shell form. Shell form wraps your process in `/bin/sh -c`, which becomes PID 1 instead of your program — so it doesn't forward `SIGTERM`, and `docker stop` ends up killing it hard after the grace period (the lifecycle from module 01).